# Dependency install

In [ ]:
!pip install -q docling "docling[asr]" semantic-chunking pymupdf requests pillow transformers python-docx python-pptx python-magic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 11.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of typer-slim to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.6/452.6 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.3/269.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [11]:
!pip install -qU langchain-huggingface langchain-core langchain torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 974.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 823.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.

# File Ingestion

In [ ]:

from docling.document_converter import DocumentConverter, AudioFormatOption
from docling.datamodel import asr_model_specs
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import AsrPipelineOptions
from docling.pipeline.asr_pipeline import AsrPipeline
from semantic_chunking import SemanticChunker
import requests
from io import BytesIO
import fitz
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from docx import Document
from pptx import Presentation
import magic
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

# function to convert in embeddings and store in vector store
def addInVectorStore(chunks):

  docs = []
  for i in range(len(chunks)):
    docs.append(Document(page_content=chunks[i], metadata={"id" : i}))

  vector_store.add_documents(docs)

# function for generating captions for image
def imageCaptioning(link):

  resp = requests.get(link)
  file_stream = BytesIO(resp.content)
  mime = magic.from_buffer(file_stream.getvalue(), mime=True)

  images = []

  if mime == "application/pdf":
    docs = fitz.open(stream=file_stream, filetype="pdf")

    for page in docs:
      for img in page.get_images(full=True):
        xref = img[0]
        base_image = docs.extract_image(xref)
        image_bytes = base_image["image"]
        image = Image.open(BytesIO(image_bytes)).convert("RGB")
        images.append(image)

  elif mime == "application/vnd.openxmlformats-officedocument.wordprocessingml.document":
    doc = Document(file_stream)

    for rel in doc.part.rels.values():
      if "image" in rel.target_ref:
        image_bytes = rel.target_part.blob
        image = Image.open(BytesIO(image_bytes)).convert("RGB")
        images.append(image)

  elif mime == "application/vnd.ms-powerpoint":
    prs = Presentation(file_stream)

    for slide in prs.slides:
      for shape in slide.shapes:
        if shape.shape_type == 13: # picture
          image_bytes = shape.image.blob
          image = Image.open(BytesIO(image_bytes)).convert("RGB")
          images.append(image)

  processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
  model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

  captions = []

  for img in images:
    inputs = processor(img, return_tensors="pt")
    out = model.generate(**inputs)
    caption = processor.decode(out[0], skip_special_tokens=True)
    captions.append(caption)

  return captions

# function for semantic chunking of texts
def semanticChunking(text):
  chunker = SemanticChunker(model_name='all-MiniLM-L6-v2', max_chunk_size=1500, similarity_threshold=0.7)
  chunks = chunker.semantic_chunk(text)

  print(f"Total chunk length : {len(chunks)}")
  print("\nSemantic Chunks:\n")
  for chunk in chunks:
    print(f"{chunk}\n")

  return chunks

# function for document extraction
def extractDocument(link):
  pdfconverter = DocumentConverter()
  doc = pdfconverter.convert(link).document
  content = doc.export_to_markdown()
  print(content)
  captions = imageCaptioning(link)

  if len(captions) > 0:
    content += "\n\n IMAGE DESCRIPTIONS : \n\n"
    for text in captions:
      content += f"{text}\n"

  chunks = semanticChunking(content)
  addInVectorStore(chunks)

# function for audio & video extraction
def mediaExtraction(link):
  pipeline_options = AsrPipelineOptions()
  pipeline_options.asr_options = asr_model_specs.WHISPER_TURBO

  mediaconverter = DocumentConverter(
       format_options = {
          InputFormat.AUDIO: AudioFormatOption(
             pipeline_cls = AsrPipeline,
             pipeline_options = pipeline_options
         )
      }
  )

  result = mediaconverter.convert(link).document
  content = result.export_to_markdown()

  chunks = semanticChunking(content)
  addInVectorStore(chunks)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vector_store = InMemoryVectorStore(embedding=embedding_model)

link = input("Enter a link : ")

if "google.com" in link:
  print("URL not allowed")
else:
  resp = requests.get(link)
  file_stream = BytesIO(resp.content)
  mime = magic.from_buffer(file_stream.getvalue(), mime=True)

  if mime == "audio/mpeg" or mime == "video/mp4":
    mediaExtraction(link)
  else:
    extractDocument(link)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total chunk length : 318

Semantic Chunks:

## A Correlative Survey on Imbalanced Breast Cancer Data Using Ensembled Oversampling Techniques and Deep Learning Algorithms

*Debaleena Datta Department of CSE Techno India Saltlake Kolkata, India leenadatta1907@gmail.com

Pradeep Kumar Mallick Department of CSE Kalinga Institute of Industrial Technology, Bhubaneswar, India

pradeep.mallickfcs@kiit.ac.in

Abstract -Inadequacy of labeled breast cancer data causes imbalance and inaccuracy in classifiers. Our proposed research aims for a computer assisted detection (CAD) system to detect maligned cancer cells using the Wisconsin Breast Cancer dataset through a novel ensembled approach that combines various oversampling methods with standard machine learning and deep learning classifiers.

The work is three-folds: (1) Use of 6 oversampling techniques: Random oversampling, Synthetic minority oversampling technique, Borderline SMOTE, k-means SMOTE, Support vector machine SMOTE, Adaptive synthetic

# Retrieval

In [13]:

import torch
from transformers import pipeline

question = input("Ask any question based on your data : ")
allDocs = vector_store.similarity_search(query, k=10)

context = ""

for doc in allDocs:
  context += f"{doc.page_content}\n"

pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.bfloat16, device_map="auto")

query = f'''
Try to summarize the context and give answer of the query from it.
If any information is not available on the context but is publically available then you can add that as well.

QUERY: {question}
CONTEXT: {context}
'''

messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who always responds based on the context provided. And if you dont find any matched context for the question then try to answer from your knowledge(only if you are confident) otherwise tell simply that you didnt found any relevant context",
    },
    {
        "role": "user",
        "content": f"{query}"
    },
]

prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])

Ask any question based on your data : what is the document tells about ?


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'max_new_tokens', 'temperature', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|system|>
You are a friendly chatbot who always responds based on the context provided. And if you dont find any matched context for the question then try to answer from your knowledge(only if you are confident) otherwise tell simply that you didnt found any relevant context</s>
<|user|>

Try to summarize the context and give answer of the query from it.
If any information is not available on the context but is publically available then you can add that as well.

QUERY: what is the document tells about ?
CONTEXT: The main component of this technique is the use of gradient-boosted decision trees. Gradient boosting decision tree approach is used by XGBoost package.
In this method, the root of the tree represents the property that best classifies the data.
The network uses backpropagation and gradient descent to select the best filters.
Artificial neural network (ANN): DL algorithms are based on ANNs which are made up of numerous interconnected nodes, each of which carries out a basic ma